The goal of this notebook is to find motifs that have passed both MS2LDA and proccess_motifs (the function) filtering and are also reproducible across runs. The two stage filtering is quite strict and demanding, but it also provides a good opportunity to explore very promising Mass2Motifs.

In [1]:

from MS2LDA.Add_On.MassQL.MassQL4MotifDB import load_motifDB, motifDB2motifs
import pickle
import tomotopy as tp
import pandas as pd
from matchms import calculate_scores
import pandas as pd
from matchms.similarity import CosineGreedy

In order to do so, everything that is needed is loaded. Basically, the motif objects and also the results from the process_motifs for each run are loaded. In order for the snippet of code below to work, make sure that the path to the needed files are correct also in your case.

In [3]:
runs = [400, 600, 800, 1000]

motifs = {}
doc2spec_maps = {}
lda_models = {}
results_sos = {}
motif_to_docs = {}
processed_ids = {}

base = "/home/ioannis/thesis_data/new_ms2lda_output/gnps_neg"

for r in runs:
    # Load motifDB and motifs
    motifDB_1, motifDB_2 = load_motifDB(f"{base}/new_negative_gnps_{r}/motifset_optimized.json")
    motifs[r] = motifDB2motifs(motifDB_2)

    # Load doc2spec_map
    with open(f"{base}/new_negative_gnps_{r}/doc2spec_map.pkl", "rb") as f:
        doc2spec_maps[r] = pickle.load(f)

    # Load LDA model
    lda_models[r] = tp.LDAModel.load(f"{base}/new_negative_gnps_{r}/ms2lda.bin")

    # Load processed results
    with open(f"{base}/results_sos_gnps_neg_{r}.pkl", "rb") as f:
        cache = pickle.load(f)
        results_sos[r] = cache["results_sos"]
        motif_to_docs[r] = cache["motif_to_docs"]
        processed_ids[r] = cache["results_sos"]["motif_ids"]

    print(f"Run {r}: {len(processed_ids[r])} motifs kept")


Run 400: 186 motifs kept
Run 600: 248 motifs kept
Run 800: 341 motifs kept
Run 1000: 392 motifs kept


process_motifs function doesn't save the matchms spectrum objects, but saves the ids of these motifs. Therefore, since the initial matchms spectrum objects are already loaded, by using the ids we can recover the spectrum objects of the motifs that passed the filtering.

In [4]:
print(motifs[400][0].metadata['motif_id'])

motif_399


In [5]:
print(processed_ids)

{400: [281, 99, 303, 238, 262, 343, 150, 144, 319, 378, 80, 77, 280, 345, 96, 66, 204, 4, 127, 12, 322, 265, 181, 218, 145, 38, 212, 232, 53, 165, 370, 103, 70, 285, 253, 182, 19, 186, 139, 36, 170, 387, 349, 230, 205, 340, 301, 275, 130, 28, 374, 339, 350, 381, 29, 227, 8, 256, 291, 309, 111, 31, 254, 175, 324, 129, 63, 59, 166, 246, 104, 171, 74, 101, 161, 234, 257, 252, 180, 224, 76, 209, 284, 219, 288, 266, 235, 258, 86, 112, 392, 228, 108, 282, 327, 43, 82, 306, 78, 192, 331, 220, 120, 278, 35, 239, 85, 41, 335, 3, 274, 332, 46, 315, 357, 5, 71, 198, 57, 384, 27, 321, 389, 286, 98, 202, 69, 251, 134, 293, 25, 174, 26, 30, 73, 149, 106, 318, 302, 6, 310, 164, 110, 313, 2, 311, 39, 40, 194, 292, 226, 100, 221, 338, 362, 237, 105, 398, 160, 206, 64, 367, 316, 196, 263, 248, 97, 102, 157, 214, 137, 231, 272, 334, 168, 16, 162, 271, 135, 125, 117, 187, 60, 268, 249, 42], 600: [518, 359, 343, 150, 488, 178, 152, 583, 32, 80, 351, 91, 34, 280, 345, 418, 66, 119, 504, 569, 415, 127, 414, 

In [6]:
print(processed_ids[400][0])

281


In [7]:
processed_motifs = {}

for r in runs:
    processed_motifs[r] = []

    for spec in motifs[r]:
        motif_id = int(spec.metadata['motif_id'].split('_')[1])
        if motif_id in processed_ids[r]:
            processed_motifs[r].append(spec)

    print(r, len(processed_motifs[r]))


400 186
600 248
800 341
1000 392


In [8]:
cosine = CosineGreedy(tolerance=0.01)

This is to check how many Mass2Motifs after MS2LDA's optimization and the process_motifs function can be found in all 4 runs, with high mean cosine similarity scores. The thresholds provided are very strict, so a relative low number of Mass2Motifs is expected. It is a good first step for isolating very promising Mass2Motifs. The seach will broaden of course later on.

In [9]:
records = []

scores_400_600 = calculate_scores(processed_motifs[400], processed_motifs[600], cosine)
scores_400_800 = calculate_scores(processed_motifs[400], processed_motifs[800], cosine)
scores_400_1000 = calculate_scores(processed_motifs[400], processed_motifs[1000], cosine)

for spec_400 in processed_motifs[400]:
    motif_id_400 = spec_400.metadata["motif_id"]

    # best match in 600
    row_600 = scores_400_600.scores_by_reference(spec_400, "CosineGreedy_score")
    if len(row_600) == 0:
        best_600_id = None
        score_600 = 0.0
    else:
        best_600_spec, best_600_score = max(row_600, key=lambda x: x[1][0])
        best_600_id = best_600_spec.metadata["motif_id"]
        score_600 = best_600_score[0]

    # best match in 800
    row_800 = scores_400_800.scores_by_reference(spec_400, "CosineGreedy_score")
    if len(row_800) == 0:
        best_800_id = None
        score_800 = 0.0
    else:
        best_800_spec, best_800_score = max(row_800, key=lambda x: x[1][0])
        best_800_id = best_800_spec.metadata["motif_id"]
        score_800 = best_800_score[0]

    # best match in 1000
    row_1000 = scores_400_1000.scores_by_reference(spec_400, "CosineGreedy_score")
    if len(row_1000) == 0:
        best_1000_id = None
        score_1000 = 0.0
    else:
        best_1000_spec, best_1000_score = max(row_1000, key=lambda x: x[1][0])
        best_1000_id = best_1000_spec.metadata["motif_id"]
        score_1000 = best_1000_score[0]

    mean_score = (score_600 + score_800 + score_1000) / 3

    records.append({
        "motif_400": motif_id_400,
        "best_600": best_600_id,
        "score_600": score_600,
        "best_800": best_800_id,
        "score_800": score_800,
        "best_1000": best_1000_id,
        "score_1000": score_1000,
        "mean_score": mean_score
    })

reproducibility_400 = pd.DataFrame(records)

In [11]:
reproducibility_400["present_in_all_runs"] = (
    (reproducibility_400["score_600"] > 0) &
    (reproducibility_400["score_800"] > 0) &
    (reproducibility_400["score_1000"] > 0)
)

reproducibility_400["high_reproducibility"] = (
    (reproducibility_400["mean_score"] >= 0.95) &
    (reproducibility_400["present_in_all_runs"])
)

In [12]:
n_high_95 = reproducibility_400["high_reproducibility"].sum()
print("Motif-groups present in all runs with mean_score ≥ 0.95:", n_high_95)

Motif-groups present in all runs with mean_score ≥ 0.95: 40


In [13]:
mid_quality = reproducibility_400[
    (reproducibility_400["present_in_all_runs"]) &
    (reproducibility_400["mean_score"] >= 0.90) &
    (reproducibility_400["mean_score"] < 0.95)
]

print(len(mid_quality))
mid_quality.head()


12


,motif_400,best_600,score_600,best_800,score_800,best_1000,score_1000,mean_score,present_in_all_runs,high_reproducibility
2,motif_303,motif_326,0.770497,motif_183,1.000000,motif_416,1.000000,0.923499,True,False
4,motif_262,motif_245,1.000000,motif_126,0.771164,motif_266,0.977162,0.916109,True,False
12,motif_280,motif_169,0.904393,motif_482,0.904320,motif_786,0.999988,0.936234,True,False
20,motif_322,motif_595,0.948656,motif_727,0.844220,motif_623,0.946893,0.913256,True,False
24,motif_145,motif_333,0.709997,motif_174,0.997232,motif_565,0.995263,0.900831,True,False


In [56]:
high_quality = reproducibility_400[reproducibility_400["high_reproducibility"]]
print(len(high_quality))
high_quality.head()

65


,motif_400,best_600,score_600,best_800,score_800,best_1000,score_1000,mean_score,present_in_all_runs,high_reproducibility
5,motif_144,motif_273,1.000000,motif_676,1.000000,motif_721,1.000000,1.000000,True,True
6,motif_207,motif_544,1.000000,motif_737,1.000000,motif_403,0.999992,0.999997,True,True
9,motif_34,motif_413,0.999939,motif_421,0.999939,motif_697,0.999942,0.999940,True,True
11,motif_280,motif_379,0.999998,motif_145,0.999980,motif_145,0.999870,0.999950,True,True
14,motif_330,motif_280,0.999897,motif_398,0.999967,motif_459,0.999967,0.999944,True,True


In [ ]:
# Save the reproducibility results to an Excel file
output_path = "/home/ioannis/thesis_data/new_ms2lda_output/msn_neg/processed_reproducible_motifs_msn_neg.xlsx"

high_quality.to_excel(output_path, index=False)

print(f"Excel file saved to: {output_path}")


Excel file saved to: /home/ioannis/thesis_data/new_ms2lda_output/msn_neg/processed_reproducible_motifs_msn_neg.xlsx


In [14]:
# Save the reproducibility results to an Excel file
output_path = "/home/ioannis/thesis_data/new_ms2lda_output/gnps_neg/processed_reproducible_motifs_gnps_neg_90.xlsx"

mid_quality.to_excel(output_path, index=False)

print(f"Excel file saved to: {output_path}")

Excel file saved to: /home/ioannis/thesis_data/new_ms2lda_output/gnps_neg/processed_reproducible_motifs_gnps_neg_90.xlsx


The next explorative step is to find how many Mass2Motifs that passed both filtering stages (MS2LDA's optimization and process_motifs function) can be found in both 400 and 600 run. Since all four MS2LDA runs were run with the same input and identical paramaters, varying only the number of Mass2Motifs, it is expected that reproducible Mass2Motifs will be found.

In [34]:
records = []

scores_400_600 = calculate_scores(processed_motifs[400], processed_motifs[600], cosine)


for spec_400 in processed_motifs[400]:
    motif_id_400 = spec_400.metadata["motif_id"]

    # best match in 600
    row_600 = scores_400_600.scores_by_reference(spec_400, "CosineGreedy_score")
    if len(row_600) == 0:
        best_600_id = None
        score_600 = 0.0
    else:
        best_600_spec, best_600_score = max(row_600, key=lambda x: x[1][0])
        best_600_id = best_600_spec.metadata["motif_id"]
        score_600 = best_600_score[0]

    records.append({
        "motif_400": motif_id_400,
        "best_600": best_600_id,
        "score_600": score_600,
    })

reproducibility_400 = pd.DataFrame(records) 

print(reproducibility_400)

     motif_400   best_600  score_600
0    motif_399  motif_186   1.000000
1     motif_79  motif_421   1.000000
2    motif_238       None   0.000000
3    motif_343  motif_359   1.000000
4    motif_154       None   0.000000
..         ...        ...        ...
168  motif_153   motif_49   0.007348
169  motif_185  motif_354   1.000000
170  motif_240  motif_341   1.000000
171  motif_307   motif_67   0.301092
172  motif_366       None   0.000000

[173 rows x 3 columns]


In [35]:
reproducibility_400["present_in_all_runs"] = (
    (reproducibility_400["score_600"] > 0))

reproducibility_400["high_reproducibility"] = (
    (reproducibility_400["score_600"] >= 0.95) &
    (reproducibility_400["present_in_all_runs"])
)

In [36]:
reproducibility_400[reproducibility_400["high_reproducibility"]]

,motif_400,best_600,score_600,present_in_all_runs,high_reproducibility
0,motif_399,motif_186,1.0,True,True
1,motif_79,motif_421,1.0,True,True
3,motif_343,motif_359,1.0,True,True
7,motif_259,motif_38,1.0,True,True
8,motif_32,motif_180,1.0,True,True
...,...,...,...,...,...
157,motif_248,motif_498,1.0,True,True
159,motif_97,motif_434,1.0,True,True
161,motif_176,motif_12,1.0,True,True
169,motif_185,motif_354,1.0,True,True


In [39]:
anchors_400_600_95 = reproducibility_400[reproducibility_400["high_reproducibility"]].copy()
anchors_400_600_95.to_excel("processed_400_600_95.xlsx", index=False)